Question: Does the systematic scan's two headline candidates (ISG15, RGS14) support individual-level prediction across cohorts?

Feature set rationale: ISG15 (2 probes: cg08469540, cg25610492) and RGS14 (5 probes: cg17509989, cg16006841, cg06060754, cg11598255, cg25875191) This 7-probe set is fixed and pre-specified; it was not chosen by searching for whichever combination scored best.

Full process summary:
- A heavily-searched multi-feature ensemble (6-11 features, chosen via nested feature-size search) was tried first. It scored 74-86% in training-cohort cross-validation but collapsed to ~50% balanced accuracy on the held-out cohort in both directions
- A batch effect was identified (16 of 17 candidate probes showed a consistent directional shift between cohorts) and corrected via per-cohort standardization
- A minimal model using only the 2 ISG15 probes, batch-corrected, no search, was tested and showed promising accuracy (below)
- Extending to all 7 ISG15+RGS14 probes was tested next, for the completeness reason stated above.
- A further round of algorithm/threshold optimization, selected via cross-validation strictly within the training cohort, was tried and did not improve on the plain default model

The final result below uses a plain, default logistic regression, no threshold tuning, on the fixed 7-probe set.

Loading data, packages

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix

data_dir = Path("/home/ethan-xiao/food-allergy-biomarkers/data")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

infant = pd.read_csv(data_dir/'candidate_features_infant.csv', index_col=0)
adolescent = pd.read_csv(data_dir/'candidate_features_adolescent.csv', index_col=0)

print(f"Infant cohort: {infant.shape[0]} samples, {infant['allergy_status'].value_counts().to_dict()}")
print(f"Adolescent cohort: {adolescent.shape[0]} samples, {adolescent['allergy_status'].value_counts().to_dict()}")

Infant cohort: 59 samples, {'allergic': 39, 'control': 20}
Adolescent cohort: 43 samples, {'allergic': 28, 'control': 15}


Batch effect correction

In [4]:
def standardize_within_cohort(df, feature_cols):
    X = df[feature_cols].copy()
    X = (X - X.mean()) / X.std()
    y = df['allergy_status'].map({'allergic': 1, 'control': 0})
    return X, y

FEATURES = [
    'cg08469540', 'cg25610492',               #ISG15
    'cg17509989', 'cg16006841', 'cg06060754',  #RGS14
    'cg11598255', 'cg25875191',                #RGS14
]

X_infant, y_infant = standardize_within_cohort(infant, FEATURES)
X_adolescent, y_adolescent = standardize_within_cohort(adolescent, FEATURES)

Building classifier

In [5]:
def bootstrap_ci(y_true, y_pred, n_bootstrap=2000, ci=95, seed=RANDOM_SEED):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rng = np.random.RandomState(seed)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        scores.append(balanced_accuracy_score(y_true[idx], y_pred[idx]))
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    return lower, upper


def run_cross_cohort(X_train, y_train, X_test, y_test, train_label, test_label):
    clf = LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED, max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    bal_acc = balanced_accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    ci_lower, ci_upper = bootstrap_ci(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"=== Train={train_label}, Test={test_label} ===")
    print(f"Balanced accuracy: {bal_acc:.3f}  (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")
    print(f"AUC: {auc:.3f}")
    print(f"Confusion matrix:\n{cm}")
    print()
    return {'balanced_accuracy': bal_acc, 'auc': auc, 'ci': (ci_lower, ci_upper)}


result_infant_to_adolescent = run_cross_cohort(
    X_infant, y_infant, X_adolescent, y_adolescent, 'Infant', 'Adolescent'
)
result_adolescent_to_infant = run_cross_cohort(
    X_adolescent, y_adolescent, X_infant, y_infant, 'Adolescent', 'Infant'
)

=== Train=Infant, Test=Adolescent ===
Balanced accuracy: 0.611  (95% CI: [0.474, 0.762])
AUC: 0.619
Confusion matrix:
[[ 6  9]
 [ 5 23]]

=== Train=Adolescent, Test=Infant ===
Balanced accuracy: 0.733  (95% CI: [0.607, 0.840])
AUC: 0.782
Confusion matrix:
[[16  4]
 [13 26]]



Using the project's two headline systematic-scan candidates (ISG15, RGS14), with a plain, untuned classifier, one cross-cohort direction shows statistically robust individual-level predictive signal and the other shows a supportive but not independently significant signal.